In [8]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [10]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [56]:
import json


def generate_dataset():
    prompt = """
        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete. 
        Add a "solution criteria" field that specifies what the correct solution should look like, and can be used to automatically grade the model's output. 
        The "solution criteria" should be a string that describes the key features of a correct solution, and can be used to automatically grade the model's output.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex",
                "solution_criteria": "Description of what a correct solution should look like"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
        * Focus on tasks that do not require writing much code and solution criteria.

        Please generate 3 objects.
    """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    
    return json.loads(text)

In [57]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Write a Python function that extracts all S3 bucket names from an AWS CloudFormation template dictionary and returns them as a list.', 'format': 'python', 'solution_criteria': "Function should iterate through the template resources, identify S3 bucket resources (type: 'AWS::S3::Bucket'), and return a list of bucket names. Should handle cases where bucket names are defined in the 'BucketName' property or auto-generated."}, {'task': "Create a JSON policy document that allows an IAM user to read objects from a specific S3 bucket named 'my-data-bucket' and list its contents, but denies all other actions.", 'format': 'json', 'solution_criteria': "Valid JSON IAM policy with correct structure including Version, Statement array, Effect set to 'Allow', Principal or Resource field targeting the S3 bucket, and Actions limited to s3:GetObject and s3:ListBucket. Should not grant permissions beyond read/list operations."}, {'task': 'Write a regular expression that matches valid AWS IAM ro

In [58]:
with open('dataset2.json', 'w') as f:
    json.dump(dataset, f, indent=2)

#### **Prompt Evaluation Pipeline**

In [59]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
        You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution against the specified solution criteria.

        Original Task:
        <task>
        {test_case["task"]}
        </task>

        Solution to Evaluate:
        <solution>
        {output}
        </solution>

        Solution Criteria:
        <criteria>
        {test_case["solution_criteria"]}
        </criteria>

        Output Format
        Provide your evaluation as a structured JSON object with the following fields, in this specific order:
        - "strengths": An array of 1-3 key strengths
        - "weaknesses": An array of 1-3 key areas for improvement
        - "reasoning": A concise explanation of your overall assessment
        - "score": A number between 1-10

        Respond with JSON. Keep your response concise and direct.
        Example response shape:
        {{
            "strengths": string[],
            "weaknesses": string[],
            "reasoning": string,
            "score": number
        }}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    
    return json.loads(eval_text)

In [60]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    
    prompt = f"""
        Please solve the following task:

        {test_case["task"]}

        * Respond only with Python, JSON, or a plain Regex
        * Do not add any comments or commentary or explanation
    """
    
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    
    return output

In [61]:
# Functions to validate the output structure
import re
import ast


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [62]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "reasoning": reasoning,
    }

In [63]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    
    results = []
    
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [64]:
with open("dataset2.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 8.166666666666666


In [65]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport json\nfrom typing import List, Any\n\ndef extract_s3_buckets(template: dict) -> List[str]:\n    \"\"\"Extract all S3 bucket names from a CloudFormation template.\"\"\"\n    bucket_names = []\n    \n    def search_dict(obj: Any) -> None:\n        if isinstance(obj, dict):\n            for key, value in obj.items():\n                if key == \"BucketName\" and isinstance(value, str):\n                    bucket_names.append(value)\n                elif isinstance(value, (dict, list)):\n                    search_dict(value)\n        elif isinstance(obj, list):\n            for item in obj:\n                if isinstance(item, (dict, list)):\n                    search_dict(item)\n    \n    search_dict(template)\n    return bucket_names\n",
    "test_case": {
      "task": "Write a Python function that extracts all S3 bucket names from an AWS CloudFormation template dictionary and returns them as a list.",
      "format": "python",
      "solution_criteria":